### SmashBot Behaviour Cloning

In [1]:
import os, random, time

import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
import torch.nn as nn
import torch.nn.functional as F

import wandb 

from data.dataset import SmashBrosDataset, BucketBatchSampler, MISC_TYPE, ACTION_TYPE, PROJECTILE_TYPE, PLAYER_TYPE, NANA_TYPE


In [2]:
PKL_DIR_TRAIN = '/home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train'
PKL_DIR_TEST = '/home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/test'

DATASET_SIZE_TRAIN = 30
DATASET_SIZE_TEST = 1
DATASET_PROCESSES = 30

BATCH_SIZE_TRAIN = 1024
BATCH_SIZE_TEST = 1024

VALIDATION_EVERY = 5000
LOG_EVERY = 100
NUM_ROUNDS = 100
EPOCHS_PER_ROUND = 2

MODEL_SAVEPATH = 'SmashBotTransformer.pt'

#### Dataset test

In [3]:
# FILES = '/home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle'
# # test_data = [pkl.path for pkl in os.scandir(PKL_DIR_TEST) if pkl.name.endswith(".pkl")]
# test_data = [os.path.join(FILES, batch) for batch in ['batch_2067.pkl', 'batch_2344.pkl', 'batch_636.pkl']]

# sampled_test_data = random.sample(test_data, 1)
# test = SmashBrosDataset(sampled_test_data)

In [4]:
# batch_size = 5

# sampler = BucketBatchSampler(test.inputs, batch_size)
# dataloader = DataLoader(test, batch_sampler=sampler)
# for i, (input, output) in enumerate(dataloader):
#     print(input.shape)  # This will print the shape of each batch
#     if i == 10: break

#### Model

In [5]:
from melee.enums import Stage, Action, Character, ProjectileType

TYPE_LIST = [MISC_TYPE, ACTION_TYPE, PROJECTILE_TYPE, PLAYER_TYPE, NANA_TYPE, -PLAYER_TYPE, -NANA_TYPE]

stage_to_index      =  {stage.value: index for index, stage in enumerate(Stage)}
action_to_index     =  {action.value: index for index, action in enumerate(Action)}
character_to_index  =  {character.value: index for index, character in enumerate(Character)}
projectile_to_index =  {projectile.value: index for index, projectile in enumerate(ProjectileType)}
type_to_index       =  {type_: index for index, type_ in enumerate(TYPE_LIST)}

class ResidualBlock(nn.Module):
    def __init__(self, model_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.norm2 = nn.LayerNorm(model_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.act1 = nn.GELU()
        self.act2 = nn.GELU()
        self.linear1 = nn.Linear(model_dim, model_dim)
        self.linear2 = nn.Linear(model_dim, model_dim)

    def forward(self, x):
        x = x + self.dropout1(self.act1(self.linear1(self.norm1(x))))
        x = x + self.dropout2(self.act2(self.linear2(self.norm2(x))))
        return x

class SmashTransformer(nn.Module):
    def __init__(self, action_dim, embed_dim=224, model_dim=384, type_embed_dim=16, nhead=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.model_dim = model_dim
        self.action_dim = action_dim
        self.embed_dim = embed_dim
        encoder_layer = nn.TransformerEncoderLayer(model_dim, nhead, model_dim*4, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Mask for player/nana state
        C = 21
        feat_indices = torch.arange(C)
        self.player_mask = (feat_indices != 1) & (feat_indices != 3)

        # Buncha embeddings for the enums
        # action and character are both part of {player/nana}state so they split the model_dim
        self.stage_embedding = nn.Embedding(len(stage_to_index), embed_dim-3) # -3 for id, distance, frame
        self.action_embedding = nn.Embedding(len(action_to_index), (embed_dim-20)//2+1) # -20 for rest of {player/nana}state. //2 for character +1 for action importance vs char
        self.character_embedding = nn.Embedding(len(character_to_index), (embed_dim-20)//2) 
        self.projectile_embedding = nn.Embedding(len(projectile_to_index), embed_dim-8) # -7 for rest of projectile state
        self.type_embedding = nn.Embedding(len(type_to_index), type_embed_dim)
        
        # Lookups for embeddings
        max_stage_val = max(stage_to_index.keys())
        stage_lookup_tensor = torch.full((max_stage_val+1,), -1)
        for stage, idx in stage_to_index.items():
            stage_lookup_tensor[stage] = idx
        self.register_buffer('stage_lookup_tensor', stage_lookup_tensor)

        max_action_val = max(action_to_index.keys())
        action_lookup_tensor = torch.full((max_action_val+1,), -1)
        for action, idx in action_to_index.items():
            action_lookup_tensor[action] = idx
        self.register_buffer('action_lookup_tensor', action_lookup_tensor)

        max_character_val = max(character_to_index.keys())
        character_lookup_tensor = torch.full((max_character_val+1,), -1)
        for character, idx in character_to_index.items():
            character_lookup_tensor[character] = idx
        self.register_buffer('character_lookup_tensor', character_lookup_tensor)
        
        max_projectile_val = max(projectile_to_index.keys())
        projectile_lookup_tensor = torch.full((max_projectile_val+1,), -1)
        for projectile, idx in projectile_to_index.items():
            projectile_lookup_tensor[projectile] = idx
        self.register_buffer('projectile_lookup_tensor', projectile_lookup_tensor)

        max_type_val = max(type_to_index.keys())
        type_lookup_tensor = torch.full((max_type_val+1,), -1)
        for type_, idx in type_to_index.items():
            type_lookup_tensor[type_] = idx
        self.register_buffer('type_lookup_tensor', type_lookup_tensor)

        # Learnable pred token sequence        
        self.pred_token = nn.Parameter(torch.randn(1, 1, model_dim))
        self.pred_sequences = 4 # learnable + misc + 2players minimum

        self.policy_head = nn.Sequential(
            ResidualBlock(self.pred_sequences*model_dim, 2*self.pred_sequences*model_dim),
            nn.LayerNorm(self.pred_sequences*model_dim),
            nn.Linear(self.pred_sequences*model_dim, action_dim)
        )

        self.embed_linear = nn.Linear(embed_dim+type_embed_dim-1, model_dim)

        self.non_embedded_player_feats = [i for i in range(C) if i not in (1, 3)]

    def calculate_loss(self, pred_action, target_action):
        # cross entropy for buttons and mse for sticks
        buttons = pred_action[:, :self.action_dim//2]
        sticks = pred_action[:, self.action_dim//2:]

        buttons_loss = F.cross_entropy(buttons, target_action[:, :self.action_dim//2])
        sticks_loss = F.mse_loss(sticks, target_action[:, self.action_dim//2:])
        total_loss = buttons_loss + sticks_loss
        return total_loss, buttons_loss, sticks_loss
        
    def forward(self, src):
        # Each of the s in S contain info related to playerstate, or nanastate, or projectiles, or misc
        # The misc info is distance (btwn players), frame, and stage.
        B, S, C = src.shape 

        all_sequence_embeddings = torch.zeros(B, S, self.embed_dim, device=src.device)

        # Process MISC_TYPE
        # misc is always 0th sequence
        stage_indices = self.stage_lookup_tensor[src[:,0,3].long()] # stage is 3rd feature
        embedded_stage = self.stage_embedding(stage_indices)
        all_sequence_embeddings[:,0,:] = torch.cat([src[:,0,:3], embedded_stage], dim=-1)

        # Process PROJECTILE_TYPE
        projectile_mask = (src[:, :, 0] == PROJECTILE_TYPE)
        if projectile_mask.any():
            proj_values = src[:,:,8][projectile_mask]
            proj_indices = self.projectile_lookup_tensor[proj_values.long()]
            embedded_projectile_type = self.projectile_embedding(proj_indices)
            projectile_rest = src[:,:,:8][projectile_mask]
            all_sequence_embeddings[projectile_mask] = torch.cat([embedded_projectile_type, projectile_rest], dim=-1)

        # Process PLAYER_TYPE / NANA_TYPE
        player_types_mask = (torch.abs(src[:, :, 0]) == PLAYER_TYPE) | (torch.abs(src[:, :, 0]) == NANA_TYPE)
        if player_types_mask.any():
            action_indices = src[:,:,1][player_types_mask]
            character_indices = src[:,:,3][player_types_mask]

            if action_indices.any():
                action_indices = self.action_lookup_tensor[action_indices.long()]
                embedded_actions = self.action_embedding(action_indices)
            if character_indices.any():
                character_indices = self.character_lookup_tensor[character_indices.long()]
                embedded_characters = self.character_embedding(character_indices)
        
            rest_features = src[:, :, self.non_embedded_player_feats][player_types_mask]
            all_sequence_embeddings[player_types_mask] = torch.cat([embedded_actions, embedded_characters, rest_features], dim=-1)
        
        type_indices = self.type_lookup_tensor[src[:,:,0].long()]
        embedded_types = self.type_embedding(type_indices)

        all_sequence_embeddings = torch.cat([embedded_types, all_sequence_embeddings[:,:,1:]], dim=-1)        
        all_sequence_embeddings = self.embed_linear(all_sequence_embeddings)

        # Add learnable pred tokens
        all_sequence_embeddings = torch.cat([self.pred_token.expand(B, 1, -1), all_sequence_embeddings], dim=1)
        
        output = self.transformer_encoder(all_sequence_embeddings)
        output = self.policy_head(output[:, :4, :].view(B, -1))
        return output

transformer = SmashTransformer(10, embed_dim=64)
print(f"Model parameters: {sum(p.numel() for p in transformer.parameters())}")

# out = transformer(input)


# for i, (input, output) in enumerate(dataloader):
#     transformer(input)

#     if i == 100: break

Model parameters: 15439982


#### Train

In [6]:
def run_validation(model):
    test_data = [pkl.path for pkl in os.scandir(PKL_DIR_TEST) if pkl.name.endswith(".pkl")]
    sampled_test_data = random.sample(test_data, DATASET_SIZE_TEST)
    num_processes = min(DATASET_PROCESSES, DATASET_SIZE_TEST)
    test_dataset = SmashBrosDataset(sampled_test_data, num_processes=num_processes)
    sampler = BucketBatchSampler(test_dataset.inputs, BATCH_SIZE_TEST)
    val_loader = DataLoader(test_dataset, batch_sampler=sampler)   
    print(f"Successfully loaded validation dataset with {len(test_dataset)} positions")

    model.eval()

    t1 = time.perf_counter()
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
            input = input.to('cuda' if torch.cuda.is_available() else 'cpu')
            target = target.to('cuda' if torch.cuda.is_available() else 'cpu')

            pred_policy = model(input)
                        
            loss, buttons_loss, sticks_loss = model.calculate_loss(pred_policy, target)
            total_loss += loss.item() * input.size(0)  # Multiply loss by batch size to get total loss for this batch
            total_count += input.size(0)  # Accumulate the total number of examples processed

    avg_loss = total_loss / total_count  # Compute average loss  
    return avg_loss, buttons_loss.item(), sticks_loss.item()


def training_round(model, train_loader, num_epochs=10, log_every=1000, validation_every=20_000):
    best_val_loss = 1000

    # Pytorch train stuffs
    optimizer = torch.optim.AdamW(model.parameters(), lr=9e-5)
    # optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50_000)
    grad_scaler = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs): 
        model.train()
        t1 = time.perf_counter()
        
        for i, (input, target) in enumerate(train_loader):
            input = input.float().to('cuda' if torch.cuda.is_available() else 'cpu')
            target = target.float().to('cuda' if torch.cuda.is_available() else 'cpu')

            # AMP with gradient clipping and lr scheduling
            with autocast():
                pred_policy = model(input)
                loss, buttons_loss, sticks_loss = model.calculate_loss(pred_policy, target)
            
            optimizer.zero_grad()
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            grad_scaler.step(optimizer)
            scale = grad_scaler.get_scale()
            grad_scaler.update()

            skip_lr_sched = scale > grad_scaler.get_scale()
            
            if not skip_lr_sched: scheduler.step()

            if i % log_every == 0:
                print(f"Epoch {epoch}, Iteration {i}, Loss: {loss}, Buttons Loss: {buttons_loss}, Sticks Loss: {sticks_loss}")
                wandb.log({
                    "lr": scheduler.get_last_lr()[0],
                    "train_loss": loss.item(),
                    "buttons_loss": buttons_loss.item(),
                    "sticks_loss": sticks_loss.item(),
                    "iter": i
                })
            
            if i % validation_every == 0 and i > 0 :
                val_loss, val_buttons_loss, val_sticks_loss = run_validation(model)
                print(f"Validation loss: {val_loss}")
                
                wandb.log({
                    "val_loss": val_loss,
                    "val_buttons_loss": val_buttons_loss,
                    "val_sticks_loss": val_sticks_loss,
                    "iter": i})

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), "best_" + MODEL_SAVEPATH)

        print(f"Epoch took {time.perf_counter()-t1} seconds ")
        torch.save(model.state_dict(), MODEL_SAVEPATH)


def run_training(num_rounds, model):
    train_data = [pkl.path for pkl in os.scandir(PKL_DIR_TRAIN) if pkl.name.endswith(".pkl")]

    wandb.init(project="smashbot", id='z8cvq8pb', resume='must')
    for round in range(num_rounds):
        print(f"Starting round {round}")
        # build dataset 
        # randomly sample dataset_size pgn files 
        t1 = time.perf_counter()
        sampled_train_data = random.sample(train_data, DATASET_SIZE_TRAIN)
        train_dataset = SmashBrosDataset(sampled_train_data, num_processes=DATASET_PROCESSES)
        sampler = BucketBatchSampler(train_dataset.inputs, BATCH_SIZE_TRAIN)
        train_loader = DataLoader(train_dataset, batch_sampler=sampler)
        print(f"Successfully loaded dataset with {len(train_dataset)} images - {time.perf_counter()-t1} seconds")
    
        training_round(model, train_loader, num_epochs=EPOCHS_PER_ROUND, log_every=LOG_EVERY, validation_every=VALIDATION_EVERY)


In [7]:
model = SmashTransformer(action_dim=10, embed_dim=224, model_dim=480, nhead=24, num_layers=6, dropout=0.05)
model = model.cuda()
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

# load weights from SAVEPATH if it exists
if os.path.exists(MODEL_SAVEPATH):
    model.load_state_dict(torch.load(MODEL_SAVEPATH))
    print("Loaded model weights from previous training session")

run_training(NUM_ROUNDS, model)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Model parameters: 24216174
Loaded model weights from previous training session


wandb: Currently logged in as: keithg33 (open_sim2real). Use `wandb login --relogin` to force relogin


Starting round 0
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_1114_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_838_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_671_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_121_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_1154_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_468_1.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_496_2.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_501_2.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_Public_Dataset_v3/pickle/train/batch_725_2.pkl...
Loading /home/kage/smashbot_workspace/dataset/Slippi_P

/home/kage/smashbot_workspace/smashvenv/lib/python3.11/site-packages/psutil/__init__.py:2017: RuntimeWarning: available memory stats couldn't be determined and was set to 0
  ret = _psplatform.virtual_memory()


: 